In [1]:
import pandas as pd

def analyze_csv():
    # Load the CSV file into a DataFrame
    csv_file = "../data/drugbank_dti_clean.csv"
    
    try:
        df = pd.read_csv(csv_file)
        
        print(f"CSV file loaded successfully: {csv_file}")
        print("=" * 50)
        
        # Get the shape (rows, columns)
        shape = df.shape
        print(f"DataFrame shape: {shape}")
        
        # Get number of rows
        num_rows = df.shape[0]
        print(f"Number of rows: {num_rows}")
        
        # Get number of columns
        num_cols = df.shape[1]
        print(f"Number of columns: {num_cols}")
        
        print("\n" + "=" * 50)
        
        # Additional useful information
        print("Column names:")
        for i, col in enumerate(df.columns):
            print(f"  {i+1}. {col}")
        
        print(f"\nFirst few rows:")
        print(df.head())
        
        print(f"\nData types:")
        print(df.dtypes)
        
        print(f"\nBasic statistics:")
        print(df.describe())
        
        return df
        
    except FileNotFoundError:
        print(f"Error: File '{csv_file}' not found in the current directory.")
        return None
    except Exception as e:
        print(f"Error reading the CSV file: {e}")
        return None

if __name__ == "__main__":
    df = analyze_csv()


CSV file loaded successfully: ../data/drugbank_dti_clean.csv
DataFrame shape: (24525, 7)
Number of rows: 24525
Number of columns: 7

Column names:
  1. drugbank_id
  2. drug_name
  3. smiles
  4. protein_uniprot_id
  5. protein_sequence
  6. organism
  7. label

First few rows:
  drugbank_id     drug_name  \
0     DB00006   Bivalirudin   
1     DB00014     Goserelin   
2     DB00014     Goserelin   
3     DB00014     Goserelin   
4     DB00027  Gramicidin D   

                                              smiles protein_uniprot_id  \
0  CC[C@H](C)[C@H](NC(=O)[C@H](CCC(O)=O)NC(=O)[C@...             P00734   
1  CC(C)C[C@H](NC(=O)[C@@H](COC(C)(C)C)NC(=O)[C@H...             P22888   
2  CC(C)C[C@H](NC(=O)[C@@H](COC(C)(C)C)NC(=O)[C@H...             P01148   
3  CC(C)C[C@H](NC(=O)[C@@H](COC(C)(C)C)NC(=O)[C@H...             P30968   
4  CC(C)C[C@@H](NC(=O)CNC(=O)[C@@H](NC=O)C(C)C)C(...             P0AC13   

                                    protein_sequence  \
0  >lcl|BSEQ0016004|Prothro

In [2]:
df.label.eq(1).all() # Check if all labels are 1 (means there is an interaction)

np.True_

Column names:
  1. drugbank_id - unique identifier for the drug
  2. drug_name - name of the drug      
  3. smiles - SMILES representation of the drug
  4. protein_uniprot_id - unique identifier for the protein
  5. protein_sequence - amino acid sequence of the protein
  6. organism - organism from which the protein is derived
  7. label - binary label indicating interaction (1 for interaction, 0 for no interaction) all values are 1



**Handle Missing Values in CSV Data**

In [3]:
df.columns

Index(['drugbank_id', 'drug_name', 'smiles', 'protein_uniprot_id',
       'protein_sequence', 'organism', 'label'],
      dtype='object')

In [4]:
#df.isnull().sum()
df.organism.value_counts()

if df.organism.isnull().sum() > 0:
    counter = df.organism.isnull().sum()
    print("There are:", counter, "missing values in the organism column.")
if df.drugbank_id.isnull().sum() > 0:
    counter = df.drugbank_id.isnull().sum()
    print("There are:", counter, "missing values in the drugbank_id column.")
if df.drug_name.isnull().sum() > 0:
    counter = df.drug_name.isnull().sum()
    print("There are:", counter, "missing values in the drug_name column.")
if df.smiles.isnull().sum() > 0:
    counter = df.smiles.isnull().sum()
    print("There are:", counter, "missing values in the smiles column.")
if df.protein_uniprot_id.isnull().sum() > 0:
    counter = df.protein_uniprot_id.isnull().sum()
    print("There are:", counter, "missing values in the protein_uniprot_id column.")
if df.protein_sequence.isnull().sum() > 0:
    counter = df.protein_sequence.isnull().sum()
    print("There are:", counter, "missing values in the protein_sequence column.")
if df.label.isnull().sum() > 0:
    counter = df.label.isnull().sum()
    print("There are:", counter, "missing values in the label column.")

df.shape


There are: 419 missing values in the organism column.


(24525, 7)

**We will drop the rows that have NA value for organism since they are not a lot.**
**And also this was what was done in the paper.**

In [5]:
df = df.dropna(subset=["organism"])
df.shape

(24106, 7)

**Detecing outliers and duplicates in the dataset**

In [6]:
df.dupes = df[(df.drugbank_id.duplicated(keep=False)) | (df.protein_uniprot_id.duplicated(keep=False))]
df.dupes.shape

/var/folders/48/r_dz_rls5y7flr7cw8l6z6s00000gn/T/ipykernel_19293/1737827187.py:1: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  df.dupes = df[(df.drugbank_id.duplicated(keep=False)) | (df.protein_uniprot_id.duplicated(keep=False))]


(23809, 7)

**Find out which protein has the most interactions with drugs**

In [7]:
# Count unique drugs that have multiple interactions
drugs_with_multiple_targets = df[df.drugbank_id.duplicated(keep=False)].drugbank_id.nunique()

# Count unique proteins that have multiple interactions  
proteins_with_multiple_drugs = df[df.protein_uniprot_id.duplicated(keep=False)].protein_uniprot_id.nunique()

print(f"Number of drugs with multiple target interactions: {drugs_with_multiple_targets}")
print(f"Number of proteins with multiple drug interactions: {proteins_with_multiple_drugs}")

#Find top 5 proteins with most interactions
top_proteins = df['protein_uniprot_id'].value_counts().head(5)
print("Top 5 proteins with most interactions:")
print(top_proteins)

#Find top 5 drugs with most interactions
top_drugs = df['drug_name'].value_counts().head(5)
print("Top 5 drugs with most interactions:")
print(top_drugs)

Number of drugs with multiple target interactions: 3355
Number of proteins with multiple drug interactions: 2549
Top 5 proteins with most interactions:
protein_uniprot_id
P14867    174
P34903    149
P24941    149
P47869    147
P14416    147
Name: count, dtype: int64
Top 5 drugs with most interactions:
drug_name
Fostamatinib    306
Artenimol       191
Copper          147
NADH            144
Zinc acetate    124
Name: count, dtype: int64


In [9]:
# Check if specific drug-protein pairs interact with each other
def check_interaction(drug_name, protein_id):
    """Check if a specific drug and protein interact"""
    interaction = df[(df['drug_name'] == drug_name) & (df['protein_uniprot_id'] == protein_id)]
    if not interaction.empty:
        print(f"✅ {drug_name} DOES interact with protein {protein_id}")
        return True
    else:
        print(f"❌ {drug_name} does NOT interact with protein {protein_id}")
        return False

# Example: Check if top drugs interact with top proteins
print("\n" + "="*60)
print("CHECKING INTERACTIONS BETWEEN TOP DRUGS AND TOP PROTEINS:")
print("="*60)

# Get the top drug and protein names/IDs
top_drug_name = top_drugs.index[0]  # Most frequent drug
top_protein_id = top_proteins.index[0]  # Most frequent protein

print(f"\nChecking if top drug '{top_drug_name}' interacts with top protein '{top_protein_id}':")
check_interaction(top_drug_name, top_protein_id)

# Check multiple combinations
print(f"\nChecking interactions between ALL top 5 drugs and top 5 proteins:")
interaction_count = 0
total_combinations = 0

for i in range(5):  # All top 5 drugs
    for j in range(5):  # All top 5 proteins
        drug = top_drugs.index[i]
        protein = top_proteins.index[j]
        total_combinations += 1
        
        print(f"Drug: {drug} | Protein: {protein}")
        if check_interaction(drug, protein):
            interaction_count += 1
        print()

print(f"SUMMARY: {interaction_count} out of {total_combinations} top drug-protein combinations have interactions")


CHECKING INTERACTIONS BETWEEN TOP DRUGS AND TOP PROTEINS:

Checking if top drug 'Fostamatinib' interacts with top protein 'P14867':
❌ Fostamatinib does NOT interact with protein P14867

Checking interactions between ALL top 5 drugs and top 5 proteins:
Drug: Fostamatinib | Protein: P14867
❌ Fostamatinib does NOT interact with protein P14867

Drug: Fostamatinib | Protein: P34903
❌ Fostamatinib does NOT interact with protein P34903

Drug: Fostamatinib | Protein: P24941
❌ Fostamatinib does NOT interact with protein P24941

Drug: Fostamatinib | Protein: P47869
❌ Fostamatinib does NOT interact with protein P47869

Drug: Fostamatinib | Protein: P14416
❌ Fostamatinib does NOT interact with protein P14416

Drug: Artenimol | Protein: P14867
❌ Artenimol does NOT interact with protein P14867

Drug: Artenimol | Protein: P34903
❌ Artenimol does NOT interact with protein P34903

Drug: Artenimol | Protein: P24941
❌ Artenimol does NOT interact with protein P24941

Drug: Artenimol | Protein: P47869
❌ A

**Correct noise or inconsistent data entries in the CSV file**

**Encode categorical variables**

**Validate cleaned data**